# Chapter 26: quality

Fresh constructed cells cover DuPont/leverage, denominator guards, GP/A, OP, accruals, same-vintage ratios, stability noise and a quality panel. The stability cell draws 100,000 replicated short samples and two 400,000-observation composite examples, costing more than hand arithmetic. Published RMW remains a disabled aggregate-spread description.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch26_quality_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. DuPont and leverage

Compare an operating improvement with a debt-funded buyback. Decompose ROE into margin, turnover and leverage; higher ROE can accompany a lower margin. Vary D/E in the leverage formula.

In [ ]:
import numpy as np
import pandas as pd
from quant_course.quality import dupont, roe_from_leverage

idx = pd.period_range("2026-06", periods=1, freq="M")
mk = lambda v: pd.DataFrame([[v]], index=idx, columns=["f"])
firms = {"before": (60.0, 1000.0, 600.0, 400.0), "X: operations improve": (90.0, 1000.0, 600.0, 400.0), "Y: debt-funded buyback": (56.25, 1000.0, 600.0, 300.0)}
table = pd.DataFrame({k: {name: float(f.iloc[0, 0]) for name, f in dupont(*[mk(v) for v in vals]).items()} for k, vals in firms.items()}).T
assert abs(table.loc["Y: debt-funded buyback", "roe"] - 0.1875) < 1e-12 and table.loc["Y: debt-funded buyback", "net_margin"] < table.loc["before", "net_margin"]
print({d: round(roe_from_leverage(0.08, 0.03, d), 4) for d in (0.0, 0.5, 1.0, 2.0)})
table.round(4)

## 2. Denominator rules

Compare ending equity, its unguarded average, an average requiring both balances positive, and a 2%-of-assets floor. A positive average of positive and negative balances can violate the intended eligibility rule.

In [ ]:
from quant_course.quality import average_balance, guarded_ratio

cols = ["steady", "shrinking", "tiny", "negative"]
ni = pd.DataFrame([[40.0] * 4], index=idx, columns=cols)
end = pd.DataFrame([[400.0, 100.0, 2.0, -30.0]], index=idx, columns=cols); prev = pd.DataFrame([[400.0, 500.0, 300.0, 100.0]], index=idx, columns=cols)
assets = pd.DataFrame([[1000.0] * 4], index=idx, columns=cols)
den = pd.DataFrame({"ending": guarded_ratio(ni, end).iloc[0], "average, unguarded": guarded_ratio(ni, average_balance(end, prev)).iloc[0],
                    "average, both > 0": guarded_ratio(ni, average_balance(end, prev, floor=0.0)).iloc[0], "ending, 2% floor": guarded_ratio(ni, end, floor=0.02 * assets).iloc[0]})
assert abs(den.loc["negative", "average, unguarded"] - 40 / 35) < 1e-12 and np.isnan(den.loc["negative", "average, both > 0"])
den.round(3)

## 3. Profitability, accruals and one-vintage ratios

Compute GP/A, OP and accruals from matching inputs. Select income and equity from one eligible version. November's same-version ROE is 0.08, unlike new income 28 divided by old equity 400.

In [ ]:
from quant_course.factor_signals import latest_available_values
from quant_course.quality import accrual_ratio, gross_profit_to_assets, operating_profitability

mk2 = lambda *v: pd.DataFrame([list(v)], index=idx, columns=["low_debt", "high_debt"])
rev, cogs, sga, interest, assets2, equity2 = mk2(1000.0, 1000.0), mk2(600.0, 600.0), mk2(200.0, 200.0), mk2(20.0, 60.0), mk2(800.0, 800.0), mk2(400.0, 400.0)
print("GP/A", gross_profit_to_assets(rev, cogs, assets2).iloc[0].to_dict(), "| OP", operating_profitability(rev, cogs, sga, interest, equity2).iloc[0].round(3).to_dict())
print("accrual (cash-backed vs not):", accrual_ratio(mk2(50.0, 50.0), mk2(80.0, 20.0), mk2(500.0, 500.0)).iloc[0].round(3).to_dict())
versions = pd.DataFrame({"security_id": ["S1", "S1"], "period_end": pd.to_datetime(["2022-12-31", "2022-12-31"]),
                         "available_at": pd.to_datetime(["2023-03-15 16:05", "2023-11-20 16:05"], utc=True), "ingested_at": pd.to_datetime(["2023-03-15 16:12", "2023-11-20 16:12"], utc=True),
                         "net_income": [40.0, 28.0], "equity": [400.0, 350.0]})
months = pd.period_range("2023-10", "2023-11", freq="M")
pit = latest_available_values(versions, months, ["net_income", "equity"])
same = guarded_ratio(pit["net_income"], pit["equity"])["S1"]
print("same-version ROE:", same.round(3).to_dict(), "| new numerator over the old denominator:", round(28 / 400, 3))
assert abs(same.iloc[0] - 0.10) < 1e-12 and abs(same.iloc[1] - 0.08) < 1e-12

## 4. Stability noise and combining dimensions

Compare short-history SD rankings when population SDs are 0.04 and 0.06. Compare sums of signals with zero versus 0.5 correlation. The uncorrelated-composite formula is a standardized Pearson identity, not a rank-IC law.

In [ ]:
rng = np.random.default_rng(11)
rev_share = {}
for n in (5, 10):
    s1 = rng.normal(0, 0.04, (100_000, n)).std(axis=1, ddof=1); s2 = rng.normal(0, 0.06, (100_000, n)).std(axis=1, ddof=1)
    rev_share[n] = round(float((s1 > s2).mean()), 3)
print("share of worlds in which the sample ranks the more stable firm as the less stable one:", rev_share)
rho1, rho2 = 0.0241, 0.0418
def composite_ic(r12):
    cov = np.array([[1.0, r12, rho1], [r12, 1.0, rho2], [rho1, rho2, 1.0]]); z1, z2, y = np.random.default_rng(5).multivariate_normal(np.zeros(3), cov, size=400_000).T
    return float(np.corrcoef(z1 + z2, y)[0, 1])
print("composite IC, uncorrelated:", round(composite_ic(0.0), 4), "vs", round((rho1 + rho2) / np.sqrt(2), 4), "| correlation 0.5:", round(composite_ic(0.5), 4))

## 5. Constructed panel

Inspect per-metric IC, leverage confounding, tiny-equity effects and GP/A plus minus-accrual composition. The negative accrual effect is specified by the DGP, not discovered in markets. Bank exclusions remain explicit.

In [ ]:
ch26 = load_code_module("ch26_quality.py")
world = ch26.quality_world()
study = ch26.quality_study(world)
ic = pd.DataFrame(study["ic_spearman"]).T[["mean", "t", "hit_rate"]]
ic.loc["composite: GP/A and minus accruals"] = [study["ic_composite_gpa_and_minus_accrual"][k] for k in ("mean", "t", "hit_rate")]
print("Pearson IC of ROE (ending equity): raw", round(study["ic_pearson_roe_ending"]["mean"], 4), "| guarded", round(study["ic_pearson_roe_guarded"]["mean"], 4))
print("leverage of the top/bottom ROE quintile:", {k: round(v, 3) for k, v in study["leverage_of_top_and_bottom_quintiles"]["ROE (average equity)"].items()}, "| all:", round(study["mean_leverage_all"], 3))
assert study["ic_composite_gpa_and_minus_accrual"]["mean"] > max(study["ic_spearman"]["GP/A (average assets)"]["mean"], study["ic_spearman"]["minus accrual ratio"]["mean"])
assert study["banks_gpa_all_nan"] is True
ic.round(4)

## 6. Published RMW: skipped

No market series or historical figures are read by the default route.

In [ ]:
if RUN_FRENCH:
    rmw = ch26.rmw_study()
    by = pd.DataFrame(rmw["by_period"]).T
    print("max drawdown", round(rmw["max_drawdown"], 4), rmw["max_drawdown_month"], "| correlations:", {k: round(v, 3) for k, v in rmw["correlation"].items()})
    assert by.loc["all", "months"] == 758 and by.loc["all", "mean_pct"] > 0
    by.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 7. Your turn

1. In a constructed copy set BETA_ACCRUAL=0 and compare minus-accrual and composite IC.
2. Raise the denominator floor to 10% of assets; which firms are excluded?
3. Delay a restatement to February 1, 2024; which version is eligible at November 2023 month-end? Keep numerator and denominator in one row.